In [0]:
%python
# Cell 1: Ingest Raw CSV from Volume to Bronze Table
CATALOG = "ipl_catalog"
SCHEMA = "ipl_schema"
VOLUME_NAME = "ipl_ds_volume"

# Note the subfolder path from your error log: archive/ipl_ball_by_ball.csv
csv_file_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/archive/ipl_ball_by_ball.csv" 

print(f"Reading raw data from: {csv_file_path}")

# Read the raw transactional CSV
df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(csv_file_path))

# Save directly as your managed Bronze Delta Table
df_raw.write \
  .mode("overwrite") \
  .format("delta") \
  .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_ipl_deliveries")

print(f"Successfully compiled: {CATALOG}.{SCHEMA}.bronze_ipl_deliveries")

In [0]:
%sql
Select * from ipl_catalog.ipl_schema.bronze_ipl_deliveries limit 100

In [0]:
%sql
-- Cell 2: Standardize and Clean Team Metadata
CREATE OR REPLACE TABLE ipl_catalog.ipl_schema.silver_ipl_deliveries AS
SELECT 
  int(match_id) as match_id,
  cast(date as date) as match_date,
  trim(season) as season,
  trim(venue) as venue,
  trim(city) as city,
  int(innings) as inning,
  
  -- Standardizing Historical Team and Franchise Changes
  CASE 
    WHEN batting_team IN ('Kings XI Punjab', 'Punjab Kings') THEN 'Punjab Kings'
    WHEN batting_team IN ('Delhi Daredevils', 'Delhi Capitals') THEN 'Delhi Capitals'
    WHEN batting_team IN ('Rising Pune Supergiant', 'Rising Pune Supergiants') THEN 'Rising Pune Supergiants'
    WHEN batting_team IN ('Deccan Chargers', 'Sunrisers Hyderabad') THEN 'Sunrisers Hyderabad'
    WHEN batting_team IN ('Royal Challengers Bengaluru', 'Royal Challengers Bangalore') THEN 'Royal Challengers Bangalore'
    ELSE trim(batting_team) 
  END as batting_team,
  
  CASE 
    WHEN bowling_team IN ('Kings XI Punjab', 'Punjab Kings') THEN 'Punjab Kings'
    WHEN bowling_team IN ('Delhi Daredevils', 'Delhi Capitals') THEN 'Delhi Capitals'
    WHEN bowling_team IN ('Rising Pune Supergiant', 'Rising Pune Supergiants') THEN 'Rising Pune Supergiants'
    WHEN bowling_team IN ('Deccan Chargers', 'Sunrisers Hyderabad') THEN 'Sunrisers Hyderabad'
    WHEN bowling_team IN ('Royal Challengers Bengaluru', 'Royal Challengers Bangalore') THEN 'Royal Challengers Bangalore'
    ELSE trim(bowling_team) 
  END as bowling_team,
  
  int(over) as over_number,
  int(ball_in_over) as ball_in_over,
  int(ball_number) as total_ball_count,
  trim(batter) as batter,
  trim(bowler) as bowler,
  trim(non_striker) as non_striker,
  int(coalesce(batter_runs, 0)) as batter_runs,
  int(coalesce(extra_runs, 0)) as extra_runs,
  int(coalesce(total_runs, 0)) as total_runs,
  int(coalesce(wides, 0)) as wides,
  int(coalesce(noballs, 0)) as noballs,
  int(coalesce(byes, 0)) as byes,
  int(coalesce(legbyes, 0)) as legbyes,
  int(coalesce(penalty, 0)) as penalty
FROM ipl_catalog.ipl_schema.bronze_ipl_deliveries
WHERE match_id IS NOT NULL;


In [0]:
%sql
Select * from ipl_catalog.ipl_schema.silver_ipl_deliveries limit 100

In [0]:
%sql
-- Cell 3: Generate Dimension Master Tables

-- 1. Build Team Dimension
CREATE OR REPLACE TABLE ipl_catalog.ipl_schema.dim_teams AS
SELECT ROW_NUMBER() OVER (ORDER BY team_name) as team_key, team_name
FROM (
  SELECT DISTINCT batting_team AS team_name FROM ipl_catalog.ipl_schema.silver_ipl_deliveries
);

-- 2. Build Player Dimension
CREATE OR REPLACE TABLE ipl_catalog.ipl_schema.dim_players AS
SELECT ROW_NUMBER() OVER (ORDER BY player_name) as player_key, player_name
FROM (
  SELECT DISTINCT batter AS player_name FROM ipl_catalog.ipl_schema.silver_ipl_deliveries
  UNION
  SELECT DISTINCT bowler AS player_name FROM ipl_catalog.ipl_schema.silver_ipl_deliveries
);

-- 3. Build Match Dimension
CREATE OR REPLACE TABLE ipl_catalog.ipl_schema.dim_matches AS
SELECT DISTINCT 
  match_id,
  match_date,
  season,
  venue,
  city
FROM ipl_catalog.ipl_schema.silver_ipl_deliveries;


In [0]:
%sql

select * from ipl_catalog.ipl_schema.dim_teams limit 100;
    

select * from ipl_catalog.ipl_schema.dim_players limit 100;
    

select * from ipl_catalog.ipl_schema.dim_matches limit 100;

In [0]:
%sql
-- Cell 4: Compile optimized Fact Table and Apply Liquid File Clustering

CREATE OR REPLACE TABLE ipl_catalog.ipl_schema.fact_deliveries AS
SELECT 
  f.match_id,
  f.inning,
  t1.team_key as batting_team_key,
  t2.team_key as bowling_team_key,
  f.over_number,
  f.ball_in_over,
  f.total_ball_count,
  p1.player_key as batter_key,
  p2.player_key as bowler_key,
  f.batter_runs,
  f.extra_runs,
  f.total_runs,
  f.wides,
  f.noballs,
  f.byes,
  f.legbyes,
  f.penalty,
  -- Engineered Flag for Bowler Wickets: Excludes run-outs, retirements, etc.
  CASE WHEN (f.total_runs - f.batter_runs - f.wides - f.noballs - f.byes - f.legbyes) > 0 THEN 1 ELSE 0 END as is_wicket
FROM ipl_catalog.ipl_schema.silver_ipl_deliveries f
JOIN ipl_catalog.ipl_schema.dim_teams t1 ON f.batting_team = t1.team_name
JOIN ipl_catalog.ipl_schema.dim_teams t2 ON f.bowling_team = t2.team_name
JOIN ipl_catalog.ipl_schema.dim_players p1 ON f.batter = p1.player_name
JOIN ipl_catalog.ipl_schema.dim_players p2 ON f.bowler = p2.player_name;

-- Physically re-cluster parquet partitions around match_id for rapid report indexing
OPTIMIZE ipl_catalog.ipl_schema.fact_deliveries ZORDER BY (match_id);


In [0]:
%sql
select * from ipl_catalog.ipl_schema.fact_deliveries limit 100

In [0]:
%sql
CREATE OR REPLACE VIEW ipl_catalog.ipl_schema.v_gold_bi_analytics AS
SELECT 
  f.match_id,
  m.season,
  m.match_date,
  m.venue,
  m.city,
  t1.team_name as batting_team,
  t2.team_name as bowling_team,
  f.over_number,
  -- Grouping overs into traditional cricket match phases
  CASE 
    WHEN f.over_number BETWEEN 0 AND 5 THEN 'Powerplay'
    WHEN f.over_number BETWEEN 6 AND 14 THEN 'Middle Overs'
    ELSE 'Death Overs'
  END as match_phase,
  p1.player_name as batsman,
  p2.player_name as bowler,
  f.batter_runs,
  f.extra_runs,
  f.total_runs,
  f.wides,
  f.noballs,
  -- Calculate legal deliveries (wides and no-balls do not count toward a bowler's completed over)
  CASE WHEN f.wides = 0 AND f.noballs = 0 THEN 1 ELSE 0 END as is_legal_ball,
  f.is_wicket
FROM ipl_catalog.ipl_schema.fact_deliveries f
JOIN ipl_catalog.ipl_schema.dim_matches m ON f.match_id = m.match_id
JOIN ipl_catalog.ipl_schema.dim_teams t1 ON f.batting_team_key = t1.team_key
JOIN ipl_catalog.ipl_schema.dim_teams t2 ON f.bowling_team_key = t2.team_key
JOIN ipl_catalog.ipl_schema.dim_players p1 ON f.batter_key = p1.player_key
JOIN ipl_catalog.ipl_schema.dim_players p2 ON f.bowler_key = p2.player_key;
